In [ ]:
import os, json, glob, shutil, zipfile, hashlib, subprocess, datetime
import numpy as np, pandas as pd, h5py
import matplotlib.pyplot as plt
from PIL import Image
from scipy.fft import dctn
from kaggle_secrets import UserSecretsClient

USER, EMAIL, REPO = "shashiranjan-jnu", "shashi17_scs@jnu.ac.in", "dpfl-braintumor"
SECRET = "kaggle_dpfl-braintumor"
TODAY = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
SEED, IMG = 2026, 128
KEEP = ["glioma", "meningioma", "notumor", "pituitary"]          # label ids 0 to 3
PRIORITY = {"figshare": 0, "sartaj": 1, "br35h": 2}

WORK = "/kaggle/working"
OUT, QC, MAT = f"{WORK}/curated", f"{WORK}/qc", f"{WORK}/mat"
for d in (OUT, QC, MAT):
    os.makedirs(d, exist_ok=True)

token = UserSecretsClient().get_secret(SECRET)
def run(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***")
    if check and r.returncode != 0:
        raise RuntimeError(out)
    return r.returncode, out

auth = f"https://{token}@github.com/{USER}/{REPO}.git"
clean_url = f"https://github.com/{USER}/{REPO}.git"
os.chdir(WORK)
run(["rm", "-rf", REPO])
run(["git", "clone", "-q", "-b", "main", auth, REPO])
run(["git", "remote", "set-url", "origin", clean_url], cwd=REPO)

raw = pd.read_csv(f"{WORK}/{REPO}/data_manifest/raw_index.csv", dtype=str)
work = raw[raw.label.isin(KEEP)].copy().reset_index(drop=True)
work["image_id"] = work.source + ":" + work.relpath
work["patient_id"] = work.patient_id.fillna("")
print(work.groupby(["source", "label"]).size().unstack(fill_value=0))
print("Working set", len(work))

In [ ]:
def find_dir(name):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if os.path.basename(root) == name:
            return root
    raise FileNotFoundError(f"Attach {name} through Add Input")

for z in glob.glob("/kaggle/input/**/brainTumorDataPublic_*.zip", recursive=True):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(MAT)

mat_paths = {}
for p in glob.glob(f"{MAT}/**/*.mat", recursive=True) + glob.glob("/kaggle/input/**/*.mat", recursive=True):
    b = os.path.basename(p)
    if b != "cvind.mat":
        mat_paths.setdefault(b, p)
assert len(mat_paths) == 3064, f"Found {len(mat_paths)} .mat files"

ROOT = {"sartaj": find_dir("brain-tumor-classification-mri"),
        "br35h": find_dir("brain-tumor-detection")}
print("Inputs located")

In [ ]:
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def load(row):
    if row.source == "figshare":
        with h5py.File(mat_paths[os.path.basename(row.relpath)], "r") as f:
            img = np.array(f["cjdata"]["image"], dtype=np.float32).T
            mask = np.array(f["cjdata"]["tumorMask"], dtype=np.uint8).T
    else:
        with Image.open(os.path.join(ROOT[row.source], row.relpath)) as im:
            img = np.asarray(im.convert("L"), dtype=np.float32)
        mask = None
    return img, mask

def to_uint8(img):
    lo, hi = np.percentile(img, (0.5, 99.5))
    return (np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)

def pad_square(a):
    h, w = a.shape
    s = max(h, w)
    out = np.zeros((s, s), a.dtype)
    out[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = a
    return out

def resize(a, n, nearest=False):
    return np.asarray(Image.fromarray(a).resize((n, n), Image.NEAREST if nearest else Image.BOX))

def phash(a8):
    c = dctn(resize(a8, 32).astype(np.float32), norm="ortho")[:8, :8].flatten()
    return int(np.packbits(c > np.median(c[1:])).view(">u8")[0])

def dhash(a8):
    s = np.asarray(Image.fromarray(a8).resize((9, 8), Image.BOX), dtype=np.int16)
    return int(np.packbits((s[:, 1:] > s[:, :-1]).flatten()).view(">u8")[0])

N = len(work)
X = np.zeros((N, IMG, IMG), np.uint8)
M = np.zeros((N, IMG, IMG), np.uint8)
ph = np.zeros(N, np.uint64)
dh = np.zeros(N, np.uint64)
arr_sha = []
for i, row in enumerate(work.itertuples()):
    img, mask = load(row)
    a8 = pad_square(to_uint8(img))
    X[i] = resize(a8, IMG)
    if mask is not None:
        M[i] = resize(pad_square(mask), IMG, nearest=True)
    ph[i], dh[i] = phash(a8), dhash(a8)
    arr_sha.append(hashlib.sha256(X[i].tobytes()).hexdigest())
    if i % 1000 == 0:
        print(i, "of", N)

work["phash"] = [f"{int(v):016x}" for v in ph]
work["dhash"] = [f"{int(v):016x}" for v in dh]
work["array_sha256"] = arr_sha
src, pid, lab = work.source.values, work.patient_id.values, work.label.values
fs_mask_ok = (M[src == "figshare"].reshape(-1, IMG * IMG).sum(1) > 0).sum()
print("Figshare slices with a non-empty 128 x 128 mask", int(fs_mask_ok), "of 3064")

In [ ]:
DMAX = 16
def popcount(x):
    return POP[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)

chunks = []
for s in range(0, N, 512):
    d = popcount(np.bitwise_xor(ph[s:s + 512, None], ph[None, :]))
    ii, jj = np.nonzero(d <= DMAX)
    keep = jj > ii + s
    chunks.append(np.stack([ii[keep] + s, jj[keep], d[ii[keep], jj[keep]]], 1))

pairs = pd.DataFrame(np.concatenate(chunks), columns=["i", "j", "d"]).astype(np.int64)

pi, pj = pairs.i.values, pairs.j.values
pairs["kind"] = np.where(src[pi] != src[pj], "cross_source",
                np.where((src[pi] == "figshare") & (pid[pi] == pid[pj]),
                         "figshare_same_patient", "within_source"))
pairs["label_match"] = lab[pi] == lab[pj]

print(pairs.groupby(["d", "kind"]).size().unstack(fill_value=0))
print("\nShare of pairs with matching labels by distance")
print(pairs.groupby("d").label_match.mean().round(3).to_string())

rng = np.random.default_rng(SEED)
for d in range(0, DMAX + 1, 2):
    sub = pairs[(pairs.d == d) & (pairs.kind != "figshare_same_patient")]
    if len(sub) == 0:
        continue
    pick = sub.iloc[rng.choice(len(sub), min(6, len(sub)), replace=False)]
    fig, ax = plt.subplots(2, len(pick), figsize=(2.2 * len(pick), 4.8))
    ax = np.asarray(ax).reshape(2, -1)
    for k, r in enumerate(pick.itertuples()):
        for row, idx in enumerate((r.i, r.j)):
            ax[row, k].imshow(X[idx], cmap="gray")
            ax[row, k].axis("off")
            ax[row, k].set_title(f"{src[idx]}\n{lab[idx]}", fontsize=7)
    fig.suptitle(f"Hamming distance {d}, {len(sub)} pairs")
    plt.tight_layout()
    fig.savefig(f"{QC}/pairs_d{d:02d}.png", dpi=80)
    plt.show()

In [ ]:
T_DUP, T_LINK = 4, 6
work = work.drop(columns=["dup_group", "unit", "action", "kept_id"], errors="ignore")

near = pairs[(pairs.d <= T_DUP) & (pairs.kind != "figshare_same_patient")]
nbrs = [[] for _ in range(N)]
for i, j, d in near[["i", "j", "d"]].itertuples(index=False):
    nbrs[i].append((j, d))
    nbrs[j].append((i, d))

order = sorted(range(N), key=lambda i: (PRIORITY[src[i]], work.relpath.iat[i]))
is_kept = np.zeros(N, bool)
action = np.full(N, "kept", dtype=object)
kept_id = np.full(N, "", dtype=object)
match_d = np.full(N, -1)

for i in order:
    hits = [(j, d) for j, d in nbrs[i] if is_kept[j]]
    if not hits:
        is_kept[i] = True
        continue
    j, d = min(hits, key=lambda t: t[1])
    kept_id[i], match_d[i] = work.image_id.iat[j], d
    if lab[i] == lab[j]:
        action[i] = "removed_duplicate"
    elif src[j] == "figshare" and src[i] != "figshare":
        action[i] = "removed_conflict_vs_figshare"
    else:
        action[i] = "removed_label_conflict"
        action[j] = "removed_label_conflict"

work["action"], work["kept_id"], work["match_d"] = action, kept_id, match_d
print(work.groupby(["source", "action"]).size().unstack(fill_value=0))
print("\nRemoved duplicates by match distance")
print(work[work.action == "removed_duplicate"].match_d.value_counts().sort_index().to_string())

conf = work[work.action.str.contains("conflict")]
print("\nConflicts by source and label")
print(conf.groupby(["source", "label", "action"]).size().to_string())

idx_of = {iid: k for k, iid in enumerate(work.image_id)}
show = conf[conf.kept_id != ""].head(12)
if len(show):
    fig, ax = plt.subplots(2, len(show), figsize=(2.2 * len(show), 4.8))
    ax = np.asarray(ax).reshape(2, -1)
    for k, (i, r) in enumerate(show.iterrows()):
        for row, idx in enumerate((i, idx_of[r.kept_id])):
            ax[row, k].imshow(X[idx], cmap="gray")
            ax[row, k].axis("off")
            ax[row, k].set_title(f"{src[idx]}\n{lab[idx]}", fontsize=7)
    fig.suptitle("Label conflicts, removed image above, matched image below")
    plt.tight_layout()
    fig.savefig(f"{QC}/label_conflicts.png", dpi=80)
    plt.show()

In [ ]:
class DSU:
    def __init__(self, n): self.p = np.arange(n)
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra








is_clean = work.action.eq("kept").values
nf = src != "figshare"
lk = pairs[(pairs.d <= T_LINK)
           & is_clean[pairs.i.values] & is_clean[pairs.j.values]
           & nf[pairs.i.values] & nf[pairs.j.values]]
dsu2 = DSU(N)
for i, j in zip(lk.i.values, lk.j.values):
    dsu2.union(i, j)
first = {}
for i in np.where(is_clean & ~nf)[0]:
    if pid[i] in first: dsu2.union(first[pid[i]], i)
    else: first[pid[i]] = i

merged_pairs = 0
for r in work[(work.source == "figshare") & (work.kept_id != "")].itertuples():
    j = idx_of[r.kept_id]
    if src[j] == "figshare" and r.patient_id != pid[j]:
        for p in (r.patient_id, pid[j]):
            if p not in first:                       # patient with no clean slices left
                first[p] = None
        a, b = first[r.patient_id], first[pid[j]]
        if a is not None and b is not None:
            dsu2.union(a, b)
            merged_pairs += 1
print("Figshare patient pairs merged through shared slices", merged_pairs)



work["unit"] = [dsu2.find(i) for i in range(N)]

cw = work[is_clean].copy()
cw_index = cw.index.values
sizes = cw.unit.value_counts()
share = sizes.iloc[0] / len(cw)
print("Units", len(sizes), "| largest", sizes.iloc[0], f"({share:.1%} of clean set)")
assert share < 0.05, "Giant split unit, lower T_LINK before continuing"

FRAC = {"test": 0.20, "val": 0.10}
rng = np.random.default_rng(SEED)
uinfo = cw.groupby("unit").agg(n=("image_id", "size"),
                               source=("source", lambda s: s.mode()[0]),
                               label=("label", lambda s: s.mode()[0]))
split_of = {}
for _, grp in uinfo.groupby(["source", "label"]):
    units = grp.index.values.copy()
    rng.shuffle(units)
    total, acc = grp.n.sum(), {"test": 0, "val": 0}
    for u in units:
        s = "test" if acc["test"] < FRAC["test"] * total else \
            "val" if acc["val"] < FRAC["val"] * total else "train"
        if s != "train":
            acc[s] += grp.at[u, "n"]
        split_of[u] = s
cw["split"] = cw.unit.map(split_of)
print(pd.crosstab([cw.source, cw.label], cw.split, margins=True))

In [ ]:
train = cw[cw.split == "train"].copy()
y = train.label.map({c: k for k, c in enumerate(KEEP)}).values

def dirichlet_partition(y, n_clients, alpha, rng, min_size=20, tries=2000):
    for _ in range(tries):
        assign = np.empty(len(y), int)
        for c in np.unique(y):
            idx = np.where(y == c)[0]
            rng.shuffle(idx)
            cuts = (np.cumsum(rng.dirichlet(alpha * np.ones(n_clients))) * len(idx)).astype(int)[:-1]
            for k, part in enumerate(np.split(idx, cuts)):
                assign[part] = k
        if np.bincount(assign, minlength=n_clients).min() >= min_size:
            return assign
    raise RuntimeError(f"No partition with min size {min_size} at alpha {alpha}, N {n_clients}")

SCEN = {"natural3": None, "iid_k10": ("iid", 10),
        "dir0.1_k10": (0.1, 10), "dir0.5_k10": (0.5, 10), "dir5.0_k10": (5.0, 10),
        "dir0.5_k3": (0.5, 3), "dir0.5_k5": (0.5, 5), "dir0.5_k15": (0.5, 15)}
for k, (name, spec) in enumerate(SCEN.items()):
    r = np.random.default_rng(SEED + k)
    if spec is None:
        train[name] = train.source.map(PRIORITY).values
    elif spec[0] == "iid":
        a = np.empty(len(y), int)
        a[r.permutation(len(y))] = np.arange(len(y)) % spec[1]
        train[name] = a
    else:
        train[name] = dirichlet_partition(y, spec[1], spec[0], r)
    print(f"{name:12s}", np.bincount(train[name]).tolist())

In [ ]:
split_arr = np.full(N, "", dtype=object)
split_arr[cw_index] = cw.split.values
near = pairs[pairs.d <= T_DUP]
si, sj = split_arr[near.i.values], split_arr[near.j.values]
fs_clean = cw[cw.source == "figshare"]
cw = cw.merge(train[["image_id"] + list(SCEN)], on="image_id", how="left")
n_train = int((cw.split == "train").sum())

checks = {
    "T_DUP": T_DUP, "T_LINK": T_LINK, "seed": SEED,
    "near_duplicate_pairs_across_splits": int(((si != "") & (sj != "") & (si != sj)).sum()),
    "figshare_patients_in_multiple_splits": int((fs_clean.groupby("patient_id").split.nunique() > 1).sum()),
    "scenario_sums_equal_train_pool": all(int(cw[s].notna().sum()) == n_train for s in SCEN),
    "clean_images": len(cw),
    "split_sizes": cw.split.value_counts().to_dict(),
    "removed": work.action.value_counts().to_dict(),
    "largest_unit_share": round(sizes.iloc[0] / len(cw), 4),
}


lnk = pairs[pairs.d <= T_LINK]
li, lj = split_arr[lnk.i.values], split_arr[lnk.j.values]
checks["lookalike_pairs_across_splits_d_le_T_LINK"] = int(((li != "") & (lj != "") & (li != lj)).sum())


pu = cw[cw.source == "figshare"].groupby("patient_id").split.first()
bad = 0
for r in work[(work.source == "figshare") & (work.kept_id != "")].itertuples():
    pj = pid[idx_of[r.kept_id]]
    if src[idx_of[r.kept_id]] == "figshare" and r.patient_id in pu and pj in pu:
        bad += int(pu[r.patient_id] != pu[pj])
checks["figshare_shared_slice_patients_across_splits"] = bad
checks["figshare_patient_pairs_merged"] = merged_pairs






print(json.dumps(checks, indent=2))

flow = []
for s in PRIORITY:
    c = cw[cw.source == s]
    flow.append(dict(source=s, raw_files=int((raw.source == s).sum()),
                     after_label_filter=int((work.source == s).sum()),
                     after_dedup=int(((work.source == s) & is_clean).sum()),
                     train=int((c.split == "train").sum()), val=int((c.split == "val").sum()),
                     test=int((c.split == "test").sum())))
flow = pd.DataFrame(flow)
print(flow.to_string(index=False))

In [ ]:
label_id = {c: k for k, c in enumerate(KEEP)}
cw["label_id"] = cw.label.map(label_id)
for s in SCEN:
    cw[s] = cw[s].astype("Int64")
splits = cw[["image_id", "source", "relpath", "label", "label_id", "patient_id", "unit", "split"] + list(SCEN)]

np.savez_compressed(f"{OUT}/brain_mri_curated_v1.npz",
                    images=X[cw_index], masks=M[cw_index],
                    labels=cw.label_id.values.astype(np.int8),
                    image_ids=cw.image_id.values.astype(str))
splits.to_csv(f"{OUT}/splits_v1.csv", index=False)


R = f"{WORK}/{REPO}"
if os.path.isdir(R):
    run(["git", "pull", "-q", "--rebase", "--autostash", auth, "main"], cwd=R)
else:
    os.chdir(WORK)
    run(["git", "clone", "-q", "-b", "main", auth, REPO])
    run(["git", "remote", "set-url", "origin", clean_url], cwd=R)



work[["image_id", "source", "sha256", "phash", "dhash", "array_sha256"]] \
    .to_csv(f"{R}/data_manifest/hashes.csv", index=False)

work[work.action != "kept"][["image_id", "source", "label", "action", "kept_id", "match_d"]] \
    .to_csv(f"{R}/data_manifest/duplicates.csv", index=False)



splits.to_csv(f"{R}/data_manifest/splits_v1.csv", index=False)
flow.to_csv(f"{R}/data_manifest/flow_counts.csv", index=False)
json.dump(checks, open(f"{R}/data_manifest/split_checks.json", "w"), indent=2)
os.makedirs(f"{R}/docs/qc", exist_ok=True)
for p in glob.glob(f"{QC}/*.png"):
    shutil.copy(p, f"{R}/docs/qc/")




mk = "## 2026-10-01 Step 1.4 deduplication and splits (revision 3)"


dec = f"""
{mk}

D18. Figshare patients sharing an identical slice within Hamming distance {T_DUP} are merged into one split unit. A shared slice implies a duplicated or misfiled patient identity, which would otherwise leak one person across splits. {merged_pairs} patient pairs were merged.
"""







for path, text in [(f"{R}/docs/decisions.md", dec),
                   (f"{R}/docs/lab_notebook.md",
                    f"\n{mk}\n- Clean images {len(cw)}, splits {checks['split_sizes']}\n- Removed {checks['removed']}\n")]:
    if mk not in open(path).read():
        with open(path, "a") as f:
            f.write(text)

run(["git", "config", "user.name", USER], cwd=R)
run(["git", "config", "user.email", EMAIL], cwd=R)
run(["git", "add", "-A"], cwd=R)
_, status = run(["git", "status", "--porcelain"], cwd=R)
if status.strip():
    run(["git", "commit", "-q", "-m", "Step 1.4 deduplication, splits, client partitions"], cwd=R)
    run(["git", "push", "-q", auth, "main"], cwd=R)
    print("Pushed to main.")

shutil.rmtree(R, ignore_errors=True)
shutil.rmtree(MAT, ignore_errors=True)

print("Done. Output folder holds curated/brain_mri_curated_v1.npz and splits_v1.csv")